In [ ]:
import os
import sys
import json
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler, Subset
import numpy as np
from tqdm import tqdm
from collections import Counter
import random
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score,
    f1_score, precision_score, recall_score, roc_auc_score,
    precision_recall_fscore_support
)
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# --- Set seeds for reproducibility ---
def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

set_seed(42)

class FixedLandmarkDataset(Dataset):
    """
    Dataset with robust global normalization and validation.
    Returns None for corrupted samples, which collate_fn will filter out.
    """
    def __init__(self, annotations_path, data_root, label_map_path, stats_path,
                 max_frames=70, top_n_classes=200):

        print(f"\n📦 Loading dataset from {annotations_path}")
        
        with open(annotations_path, 'r') as f: self.annotations = json.load(f)
        with open(label_map_path, 'r') as f: full_label_map = json.load(f)
        with open(stats_path, 'r') as f: stats = json.load(f)
        
        self.data_root = data_root
        self.max_frames = max_frames
        self.min_frames = 5
        
        # Feature dimensions
        self.spatial_dim = 1742
        self.input_dim = self.spatial_dim * 2  # spatial + temporal
        
        # --- Normalization Stats ---
        self.mean = torch.tensor(stats['spatial_mean'] + stats['temporal_mean'], dtype=torch.float32)
        self.std = torch.tensor(stats['spatial_std'] + stats['temporal_std'], dtype=torch.float32)
        self.std[self.std < 1e-6] = 1.0 
        print("  ✅ Loaded global normalization stats.")

        # --- Class mapping ---
        all_glosses = sorted(full_label_map.keys(), key=lambda g: full_label_map[g])
        selected_glosses = all_glosses[:top_n_classes]
        self.gloss_to_idx = {gloss: i for i, gloss in enumerate(selected_glosses)}
        self.idx_to_gloss = {i: gloss for gloss, i in self.gloss_to_idx.items()}
        self.num_classes = len(self.gloss_to_idx)
        
        # --- Build samples list ---
        self.samples = []
        for entry in self.annotations:
            gloss = entry['gloss']
            if gloss not in self.gloss_to_idx: continue
            label_idx = self.gloss_to_idx[gloss]
            for instance in entry.get('instances', []):
                video_id = instance.get('video_id')
                if not video_id: continue
                path = os.path.join(self.data_root, video_id, 'landmarks.json')
                if os.path.exists(path):
                    self.samples.append({'path': path, 'label_idx': label_idx, 'video_id': video_id})
        
        print(f"  ✅ Found {len(self.samples)} potential samples.")
        self.class_counts = Counter([s['label_idx'] for s in self.samples])

    def __len__(self):
        return len(self.samples)

    def _extract_spatial_features(self, frame):
        if not isinstance(frame, dict): return None
        if 'left_hand_engineered' not in frame or 'right_hand_engineered' not in frame: return None

        def safe_get(key, size):
            data = np.array(frame.get(key, []), dtype=np.float32).flatten()
            if len(data) > size: data = data[:size]
            elif len(data) < size: data = np.pad(data, (0, size - len(data)))
            return data
        
        return np.concatenate([
            safe_get('pose', 132), safe_get('left_hand', 84), safe_get('right_hand', 84),
            safe_get('face', 1404), safe_get('left_hand_engineered', 19), safe_get('right_hand_engineered', 19)
        ])
    
    def __getitem__(self, idx):
        sample = self.samples[idx]
        landmarks_path = sample['path']
        
        try:
            with open(landmarks_path, 'r') as f: frames = json.load(f)
            if not isinstance(frames, list) or len(frames) < self.min_frames: return None
        except:
            return None

        spatial_features_list = [self._extract_spatial_features(frame) for frame in frames]
        if any(f is None for f in spatial_features_list): return None

        spatial = np.array(spatial_features_list, dtype=np.float32)
        
        if np.isnan(spatial).any() or np.isinf(spatial).any(): return None
            
        temporal = np.diff(spatial, axis=0, prepend=spatial[0:1])
        features = np.concatenate([spatial, temporal], axis=1)

        if len(features) != self.max_frames:
             indices = np.linspace(0, len(features)-1, self.max_frames, dtype=int)
             features = features[indices]
        
        x = torch.tensor(features, dtype=torch.float32)
        x = (x - self.mean) / self.std
        
        if torch.isnan(x).any() or torch.isinf(x).any(): return None
            
        return x, sample['label_idx']

def collate_fn(batch):
    batch = [item for item in batch if item is not None]
    if not batch: return None, None
    seqs, lbls = zip(*batch)
    return torch.stack(seqs), torch.tensor(lbls, dtype=torch.long)


# class SimplifiedSignModel(nn.Module):
#     """A simpler but robust LSTM-based model."""
#     def __init__(self, input_dim, num_classes, hidden_dim=384):
#         super().__init__()
#         print(f"\n🏗  Building SimplifiedSignModel:")
#         print(f"   Input dim: {input_dim}, Hidden dim: {hidden_dim}, Output classes: {num_classes}")
        
#         self.frame_encoder = nn.Sequential(
#             nn.Linear(input_dim, hidden_dim), nn.LayerNorm(hidden_dim),
#             nn.ReLU(), nn.Dropout(0.3),
#             nn.Linear(hidden_dim, hidden_dim), nn.LayerNorm(hidden_dim),
#             nn.ReLU(), nn.Dropout(0.3)
#         )
#         self.temporal = nn.LSTM(
#             hidden_dim, hidden_dim // 2, num_layers=2, batch_first=True,
#             dropout=0.3, bidirectional=True
#         )
#         self.attention = nn.Sequential(nn.Linear(hidden_dim, 64), nn.Tanh(), nn.Linear(64, 1))
#         self.classifier = nn.Sequential(
#             nn.Linear(hidden_dim, 256), nn.LayerNorm(256),
#             nn.ReLU(), nn.Dropout(0.5),
#             nn.Linear(256, num_classes)
#         )
#         self._init_weights()
#         print(f"   Total parameters: {sum(p.numel() for p in self.parameters()):,}")

#     def _init_weights(self):
#         for m in self.modules():
#             if isinstance(m, nn.Linear):
#                 nn.init.xavier_uniform_(m.weight)
#                 if m.bias is not None: nn.init.constant_(m.bias, 0)
#             elif isinstance(m, nn.LSTM):
#                 for name, param in m.named_parameters():
#                     if 'weight' in name: nn.init.xavier_uniform_(param)
#                     elif 'bias' in name: nn.init.constant_(param, 0)
    
#     def forward(self, x):
#         B, T, D = x.shape
#         x_flat = x.view(B * T, D)
#         features = self.frame_encoder(x_flat).view(B, T, -1)
#         lstm_out, _ = self.temporal(features)
#         attention_weights = F.softmax(self.attention(lstm_out), dim=1)
#         pooled = torch.sum(lstm_out * attention_weights, dim=1)
#         return self.classifier(pooled)


# class LSTMTransformerModel(nn.Module):
#     """
#     A hybrid model combining an LSTM for initial temporal feature extraction
#     followed by a Transformer Encoder for advanced sequence modeling via self-attention.
#     """
#     def __init__(self, input_dim: int, num_classes: int, hidden_dim: int = 384, nhead: int = 8, num_transformer_layers: int = 1):
#         """
#         Args:
#             input_dim: The dimension of the input features (D in B x T x D).
#             num_classes: The number of output classes (signs).
#             hidden_dim: The feature dimension used throughout the model (d_model for Transformer).
#             nhead: The number of attention heads in the Transformer Encoder.
#             num_transformer_layers: The number of Transformer Encoder layers to stack.
#         """
#         super().__init__()
#         print(f"\n🏗  Building LSTMTransformerModel:")
#         print(f"  Input dim: {input_dim}, Hidden dim: {hidden_dim}, Output classes: {num_classes}")

#         # 1. Frame Encoder (Per-Frame Feature Projection)
#         # Projects the high-dimensional frame features (D) to the model's internal hidden_dim (d_model).
#         self.frame_encoder = nn.Sequential(
#             nn.Linear(input_dim, hidden_dim), 
#             nn.LayerNorm(hidden_dim),
#             nn.GELU(), 
#             nn.Dropout(0.1),
#         )

#         # 2. Positional Encoding
#         # Adds temporal information to the features before the Transformer.
#         self.positional_encoding = nn.Parameter(torch.zeros(1, 256, hidden_dim)) # Max sequence length of 256

#         # 3. Temporal LSTM
#         # Bidirectional LSTM captures local temporal dependencies.
#         # Output dim is hidden_dim (hidden_dim // 2 * 2 for bidirectional)
#         self.temporal_lstm = nn.LSTM(
#             input_size=hidden_dim, 
#             hidden_size=hidden_dim // 2, 
#             num_layers=2, 
#             batch_first=True,
#             dropout=0.1, 
#             bidirectional=True
#         )

#         # 4. Transformer Encoder
#         # The core Transformer layer for global context modeling via self-attention.
#         transformer_layer = nn.TransformerEncoderLayer(
#             d_model=hidden_dim, 
#             nhead=nhead, 
#             dim_feedforward=hidden_dim * 4,
#             dropout=0.1, 
#             batch_first=True
#         )
#         self.transformer_encoder = nn.TransformerEncoder(
#             encoder_layer=transformer_layer, 
#             num_layers=num_transformer_layers
#         )

#         # 5. Classifier (Uses a simple mean pool over the final sequence)
#         self.classifier = nn.Sequential(
#             nn.Linear(hidden_dim, 256), 
#             nn.LayerNorm(256),
#             nn.GELU(), 
#             nn.Dropout(0.5),
#             nn.Linear(256, num_classes)
#         )
        
#         self._init_weights()
#         print(f"  Total parameters: {sum(p.numel() for p in self.parameters()):,}")


#     def _init_weights(self):
#         # A simple initialization scheme for all Linear layers
#         for m in self.modules():
#             if isinstance(m, nn.Linear):
#                 nn.init.xavier_uniform_(m.weight)
#                 if m.bias is not None: nn.init.constant_(m.bias, 0)
#             elif isinstance(m, nn.LayerNorm):
#                 nn.init.constant_(m.bias, 0)
#                 nn.init.constant_(m.weight, 1.0)


#     def forward(self, x: torch.Tensor) -> torch.Tensor:
#         """
#         Args:
#             x: A tensor of shape (B, T, D), where B=Batch, T=Time/Frames, D=Feature Dim.
#         Returns:
#             A tensor of shape (B, num_classes).
#         """
#         B, T, D = x.shape
        
#         # 1. Frame Encoding: (B*T, D) -> (B*T, H) -> (B, T, H)
#         # Project raw features to hidden_dim
#         features = self.frame_encoder(x.view(B * T, D)).view(B, T, -1)
        
#         # 2. Positional Encoding: Add temporal signal (up to T_max=256)
#         # Slice the pre-computed positional embeddings to match the current T
#         features = features + self.positional_encoding[:, :T, :]
        
#         # 3. Temporal LSTM: (B, T, H) -> (B, T, H)
#         # Pass features through the LSTM
#         lstm_out, _ = self.temporal_lstm(features)
        
#         # 4. Transformer Encoder: (B, T, H) -> (B, T, H)
#         # Use a simple mask to handle padded zeros (assuming T < 256 and padding)
#         # Note: A proper padding mask should be computed based on the sequence length. 
#         # For simplicity here, we assume inputs are already correctly padded/truncated.
#         transformer_out = self.transformer_encoder(lstm_out)
        
#         # 5. Pooling & Classification: (B, T, H) -> (B, H) -> (B, num_classes)
#         # Global Average Pooling (or another pooling method like Attention Pooling)
#         # We use a simple mean pool here.
#         pooled = torch.mean(transformer_out, dim=1) 
        
#         return self.classifier(pooled)



class StackedBiLSTMTransformerModel(nn.Module):
    """
    A hybrid model combining a stacked Bidirectional LSTM for local temporal
    feature extraction followed by a Transformer Encoder for global sequence modeling.
    """
    def __init__(self, 
                 input_dim: int, 
                 num_classes: int, 
                 hidden_dim: int = 384, 
                 nhead: int = 8, 
                 num_lstm_layers: int = 2,  # <-- Added to control LSTM stack depth
                 num_transformer_layers: int = 1):
        """
        Args:
            input_dim: The dimension of the input features (D in B x T x D).
            num_classes: The number of output classes (signs).
            hidden_dim: The feature dimension used throughout the model (d_model).
            nhead: The number of attention heads in the Transformer Encoder.
            num_lstm_layers: The number of layers in the stacked BiLSTM.
            num_transformer_layers: The number of Transformer Encoder layers.
        """
        super().__init__()
        print(f"\n🏗  Building StackedBiLSTMTransformerModel:")
        print(f"  Input dim: {input_dim}, Hidden dim: {hidden_dim}, Output classes: {num_classes}")
        print(f"  LSTM Layers: {num_lstm_layers}, Transformer Layers: {num_transformer_layers}, Heads: {nhead}")

        # 1. Frame Encoder (Per-Frame Feature Projection)
        # Projects input_dim (e.g., 1024) to the model's hidden_dim (e.g., 384)
        self.frame_encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim), 
            nn.LayerNorm(hidden_dim),
            nn.GELU(), 
            nn.Dropout(0.1),
        )

        # 2. Positional Encoding
        # Learnable positional embeddings for the Transformer
        self.positional_encoding = nn.Parameter(torch.zeros(1, 256, hidden_dim)) # Max seq length 256

        # 3. Stacked Bidirectional LSTM
        # Captures local temporal patterns.
        # Note: dropout is only applied between LSTM layers if num_lstm_layers > 1
        lstm_dropout = 0.1 if num_lstm_layers > 1 else 0.0
        self.temporal_lstm = nn.LSTM(
            input_size=hidden_dim, 
            hidden_size=hidden_dim // 2,  # Output is hidden_dim // 2 * 2 (bidirectional) = hidden_dim
            num_layers=num_lstm_layers,    # <-- Use the new parameter here
            batch_first=True,
            dropout=lstm_dropout, 
            bidirectional=True             # <-- This makes it a BiLSTM
        )

        # 4. Transformer Encoder
        # Applies self-attention to model global dependencies
        transformer_layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim, 
            nhead=nhead, 
            dim_feedforward=hidden_dim * 4,
            dropout=0.1, 
            activation="gelu", # Switched to GELU to match other activations
            batch_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(
            encoder_layer=transformer_layer, 
            num_layers=num_transformer_layers
        )

        # 5. Classifier Head
        # Pools the sequence and maps to output classes
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, 256), 
            nn.LayerNorm(256),
            nn.GELU(), 
            nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )
        
        self._init_weights()
        print(f"  Total parameters: {sum(p.numel() for p in self.parameters()):,}")


    def _init_weights(self):
        # Initialize weights
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None: nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.LayerNorm):
                nn.init.constant_(m.bias, 0)
                nn.init.constant_(m.weight, 1.0)
        
        # Initialize positional encoding
        nn.init.normal_(self.positional_encoding, std=0.02)


    def forward(self, x: torch.Tensor, src_key_padding_mask: torch.Tensor = None) -> torch.Tensor:
        """
        Args:
            x: A tensor of shape (B, T, D).
            src_key_padding_mask: (Optional) A bool tensor of shape (B, T) 
                                 where True indicates a padded element.
        Returns:
            A tensor of shape (B, num_classes).
        """
        B, T, D = x.shape
        
        # 1. Frame Encoding: (B, T, D) -> (B, T, H)
        features = self.frame_encoder(x)
        
        # 2. Positional Encoding: (B, T, H)
        if T > self.positional_encoding.shape[1]:
             raise ValueError(f"Input sequence length ({T}) exceeds max positional encoding length ({self.positional_encoding.shape[1]})")
        features = features + self.positional_encoding[:, :T, :]
        
        # 3. Temporal LSTM: (B, T, H) -> (B, T, H)
        # The LSTM processes the sequence, capturing local dependencies
        lstm_out, _ = self.temporal_lstm(features)
        
        # 4. Transformer Encoder: (B, T, H) -> (B, T, H)
        # The Transformer refines the features using global self-attention
        # We pass the padding mask to the transformer
        transformer_out = self.transformer_encoder(
            lstm_out, 
            src_key_padding_mask=src_key_padding_mask
        )
        
        # 5. Pooling & Classification: (B, T, H) -> (B, H) -> (B, num_classes)
        
        # --- Start: Masked Average Pooling ---
        # This is a more robust pooling method than simple torch.mean()
        # if you are using padding masks.
        if src_key_padding_mask is not None:
            # Invert mask: True for non-padded, False for padded
            mask = ~src_key_padding_mask.unsqueeze(-1) # Shape (B, T, 1)
            # Zero out padded values
            masked_output = transformer_out * mask
            # Sum non-padded values
            summed = torch.sum(masked_output, dim=1) # Shape (B, H)
            # Count non-padded values
            count = mask.sum(dim=1).clamp(min=1e-9) # Shape (B, 1)
            # Calculate mean
            pooled = summed / count
        else:
            # Fallback to simple mean pooling if no mask is provided
            pooled = torch.mean(transformer_out, dim=1) 
        # --- End: Masked Average Pooling ---
        
        return self.classifier(pooled)
    
    

def sanity_check_overfit(model_class, model_args, train_loader, device, max_epochs=200):
    print(f"\n{'='*60}\n🧪 SANITY CHECK: Attempting to overfit a single batch\n{'='*60}")
    model = model_class(**model_args).to(device)

    # Find first valid batch robustly
    single_batch = None
    for seqs, lbls in train_loader:
        if seqs is None:
            continue
        single_batch = (seqs, lbls)
        break

    if single_batch is None:
        print("❌ Could not load a valid batch!"); return False
    
    seqs, lbls = single_batch[0].to(device), single_batch[1].to(device)
    print(f"   Batch size: {seqs.shape[0]}, Unique labels: {len(torch.unique(lbls))}")
    
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()
    
    for epoch in range(max_epochs):
        model.train()
        optimizer.zero_grad()
        outputs = model(seqs)
        loss = criterion(outputs, lbls)
        if torch.isnan(loss).any().item():
            print("   ⚠️ NaN loss detected; skipping step.")
            continue
        loss.backward()
        optimizer.step()
        
        if (epoch + 1) % 20 == 0:
            acc = (outputs.argmax(1) == lbls).float().mean().item() * 100
            print(f"   Epoch {epoch+1:3d}: Loss={loss.item():.4f}, Acc={acc:.2f}%")
            if acc > 95:
                print(f"\n   ✅ SUCCESS! Overfitted in {epoch+1} epochs. Model can learn.")
                return True
    
    print(f"\n   ❌ FAILURE! Could not overfit. There is a fundamental issue.")
    return False

def compute_comprehensive_metrics(all_preds, all_labels, num_classes, epoch, phase='Val'):
    """Compute and print all classification metrics"""
    print(f"\n{'='*70}")
    print(f"📊 {phase} METRICS - Epoch {epoch}")
    print(f"{'='*70}")
    
    # Basic metrics
    accuracy = accuracy_score(all_labels, all_preds)
    
    # Per-class metrics with zero_division handling
    precision_macro = precision_score(all_labels, all_preds, average='macro', zero_division=0)
    recall_macro = recall_score(all_labels, all_preds, average='macro', zero_division=0)
    f1_macro = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    
    precision_weighted = precision_score(all_labels, all_preds, average='weighted', zero_division=0)
    recall_weighted = recall_score(all_labels, all_preds, average='weighted', zero_division=0)
    f1_weighted = f1_score(all_labels, all_preds, average='weighted', zero_division=0)
    
    print(f"\n📈 Overall Metrics:")
    print(f"   Accuracy:           {accuracy*100:.2f}%")
    print(f"\n   Macro Averages:")
    print(f"   - Precision:        {precision_macro*100:.2f}%")
    print(f"   - Recall:           {recall_macro*100:.2f}%")
    print(f"\n   Weighted Averages:")
    print(f"   - Precision:        {precision_weighted*100:.2f}%")
    print(f"   - Recall:           {recall_weighted*100:.2f}%")
    print(f"\n   - F1-Score (Macro): {f1_macro*100:.2f}%")
    print(f"   - F1-Score (Weighted): {f1_weighted*100:.2f}%")
    
    # Confusion Matrix Statistics
    cm = confusion_matrix(all_labels, all_preds)
    print(f"\n📊 Confusion Matrix Statistics:")
    print(f"   True Positives:     {np.diag(cm).sum()}")
    print(f"   Total Predictions:  {cm.sum()}")
    
    metrics_dict = {
        'accuracy': accuracy,
        'precision_macro': precision_macro,
        'recall_macro': recall_macro,
        'f1_macro': f1_macro,
        'precision_weighted': precision_weighted,
        'recall_weighted': recall_weighted,
        'f1_weighted': f1_weighted,
        'confusion_matrix': cm
    }
    
    return metrics_dict

def plot_confusion_matrix(cm, epoch, phase='Val', save_path='confusion_matrix.png', top_k=50):
    """Plot and save confusion matrix (showing top K classes for readability)"""
    # For large number of classes, show only top K most frequent
    if cm.shape[0] > top_k:
        row_sums = cm.sum(axis=1)
        top_indices = np.argsort(row_sums)[-top_k:]
        cm_subset = cm[np.ix_(top_indices, top_indices)]
    else:
        cm_subset = cm
    
    plt.figure(figsize=(12, 10))
    sns.heatmap(cm_subset, annot=False, fmt='d', cmap='Blues', cbar_kws={'label': 'Count'})
    plt.title(f'{phase} Confusion Matrix - Epoch {epoch}\n(Showing {"top " + str(top_k) if cm.shape[0] > top_k else "all"} classes)', fontsize=14)
    plt.ylabel('True Label', fontsize=12)
    plt.xlabel('Predicted Label', fontsize=12)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"   💾 Confusion matrix saved to: {save_path}")

def plot_metrics_history(history, save_path='training_metrics.png'):
    """Plot training history"""
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    fig.suptitle('Training History', fontsize=16, fontweight='bold')
    
    metrics = [
        ('accuracy', 'Accuracy', '%'),
        ('f1_macro', 'F1-Score (Macro)', '%'),
        ('precision_macro', 'Precision (Macro)', '%'),
        ('recall_macro', 'Recall (Macro)', '%'),
        ('loss', 'Loss', ''),
        ('f1_weighted', 'F1-Score (Weighted)', '%')
    ]
    
    for idx, (metric, title, unit) in enumerate(metrics):
        ax = axes[idx // 3, idx % 3]
        
        train_key = f'train_{metric}'
        val_key = f'val_{metric}'
        
        if train_key in history:
            epochs = range(1, len(history[train_key]) + 1)
            train_vals = [v * 100 if unit == '%' and v <= 1 else v for v in history[train_key]]
            val_vals = [v * 100 if unit == '%' and v <= 1 else v for v in history[val_key]]
            
            ax.plot(epochs, train_vals, 'b-o', label='Train', linewidth=2, markersize=4)
            ax.plot(epochs, val_vals, 'r-s', label='Val', linewidth=2, markersize=4)
            ax.set_xlabel('Epoch', fontsize=10)
            ax.set_ylabel(f'{title} {unit}', fontsize=10)
            ax.set_title(title, fontsize=12, fontweight='bold')
            ax.legend(loc='best')
            ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"   💾 Training history saved to: {save_path}")

def evaluate_model(model, loader, device, criterion, num_classes, epoch, phase='Val'):
    """Evaluate model and return comprehensive metrics"""
    model.eval()
    all_preds = []
    all_labels = []
    total_loss = 0
    batches = 0
    
    with torch.no_grad():
        for seqs, lbls in tqdm(loader, desc=f"{phase} Evaluation", ncols=100):
            if seqs is None: continue
            seqs, lbls = seqs.to(device), lbls.to(device)
            outputs = model(seqs)
            loss = criterion(outputs, lbls)
            total_loss += loss.item()
            preds = outputs.argmax(1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(lbls.cpu().numpy())
            batches += 1
    
    avg_loss = total_loss / max(1, batches)
    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)
    
    metrics = compute_comprehensive_metrics(all_preds, all_labels, num_classes, epoch, phase)
    metrics['loss'] = avg_loss
    
    return metrics

def train_model(model, train_loader, val_loader, device, epochs, save_path, num_classes):
    optimizer = torch.optim.AdamW(model.parameters(), lr=5e-4, weight_decay=1e-2)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=5, verbose=True)
    criterion = nn.CrossEntropyLoss()
    best_val_acc = 0.0
    best_val_f1 = 0.0
    patience_counter = 0
    patience_limit = 15
    
    # History tracking
    history = {
        'train_loss': [], 'train_accuracy': [], 'train_f1_macro': [], 
        'train_precision_macro': [], 'train_recall_macro': [], 'train_f1_weighted': [],
        'val_loss': [], 'val_accuracy': [], 'val_f1_macro': [],
        'val_precision_macro': [], 'val_recall_macro': [], 'val_f1_weighted': []
    }

    for epoch in range(epochs):
        print(f"\n{'='*70}")
        print(f"🚀 Epoch {epoch+1}/{epochs}")
        print(f"{'='*70}")
        
        # Training phase
        model.train()
        train_preds = []
        train_labels = []
        train_loss = 0
        train_batches = 0
        
        for seqs, lbls in tqdm(train_loader, desc="Training", ncols=100):
            if seqs is None: continue
            seqs, lbls = seqs.to(device), lbls.to(device)
            optimizer.zero_grad()
            outputs = model(seqs)
            loss = criterion(outputs, lbls)
            if torch.isnan(loss).any().item(): 
                print("   ⚠️ NaN loss detected during training; skipping batch.")
                continue
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()
            
            train_loss += loss.item()
            preds = outputs.argmax(1)
            train_preds.extend(preds.cpu().numpy())
            train_labels.extend(lbls.cpu().numpy())
            train_batches += 1
        
        # Compute training metrics
        train_preds = np.array(train_preds)
        train_labels = np.array(train_labels)
        train_metrics = compute_comprehensive_metrics(train_preds, train_labels, num_classes, epoch+1, 'Train')
        train_metrics['loss'] = train_loss / max(1, train_batches)
        
        # Validation phase
        val_metrics = evaluate_model(model, val_loader, device, criterion, num_classes, epoch+1, 'Val')
        
        # Update history
        for key in ['loss', 'accuracy', 'f1_macro', 'precision_macro', 'recall_macro', 'f1_weighted']:
            history[f'train_{key}'].append(train_metrics[key])
            history[f'val_{key}'].append(val_metrics[key])
        
        print(f"\n📊 Epoch {epoch+1} Summary:")
        print(f"   Train -> Loss: {train_metrics['loss']:.4f}, Acc: {train_metrics['accuracy']*100:.2f}%, F1: {train_metrics['f1_macro']*100:.2f}%")
        print(f"   Val   -> Loss: {val_metrics['loss']:.4f}, Acc: {val_metrics['accuracy']*100:.2f}%, F1: {val_metrics['f1_macro']*100:.2f}%")
        
        scheduler.step(val_metrics['accuracy'])
        
        # Save best model
        if val_metrics['accuracy'] > best_val_acc:
            best_val_acc = val_metrics['accuracy']
            best_val_f1 = val_metrics['f1_macro']
            patience_counter = 0
            torch.save({
                'model_state_dict': model.state_dict(),
                'epoch': epoch + 1,
                'val_acc': val_metrics['accuracy'],
                'val_f1': val_metrics['f1_macro'],
                'train_metrics': train_metrics,
                'val_metrics': val_metrics
            }, save_path)
            print(f"✅ New best model saved! Val Acc: {val_metrics['accuracy']*100:.2f}%, Val F1: {val_metrics['f1_macro']*100:.2f}%")
            
            # Plot confusion matrix for best model
            plot_confusion_matrix(val_metrics['confusion_matrix'], epoch+1, 'Val', 
                                f'confusion_matrix_epoch_{epoch+1}.png')
        else:
            patience_counter += 1
            if patience_counter >= patience_limit:
                print("🛑 Early stopping.")
                break
    
    # Plot final training history
    plot_metrics_history(history, 'training_history.png')
    
    print(f"\n{'='*70}")
    print(f"🏆 TRAINING COMPLETE!")
    print(f"{'='*70}")
    print(f"   Best Validation Accuracy: {best_val_acc*100:.2f}%")
    print(f"   Best Validation F1-Score: {best_val_f1*100:.2f}%")
    
    return best_val_acc, best_val_f1, history

def main():
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f"Using device: {device}")
    
    config = {
        'full_dataset_ann': r'D:\Balanced_20_Frames_Augmented\train_final.json',
        'full_dataset_root': r'D:\Balanced_20_Frames_Augmented\Train',
        'label_map': r'D:\Balanced_20_Frames_Augmented\label_map_final.json',
        'stats_file': r'D:\Balanced_20_Frames_Augmented\stats.json',
        'batch_size': 32,
        'epochs': 63,  # Updated to 30
        'top_n': 100,
        'val_split': 0.2
    }

    # --- Load ONE Dataset and Split It ---
    full_dataset = FixedLandmarkDataset(
        config['full_dataset_ann'], config['full_dataset_root'], config['label_map'], 
        config['stats_file'], top_n_classes=config['top_n']
    )
    
    # --- Create 80/20 Split ---
    print(f"\n🔪 Splitting data into {1-config['val_split']:.0%}/{config['val_split']:.0%} train/val sets...")
    dataset_size = len(full_dataset)
    val_size = int(dataset_size * config['val_split'])
    train_size = dataset_size - val_size
    train_dataset, val_dataset = torch.utils.data.random_split(full_dataset, [train_size, val_size])
    print(f"   Train samples: {len(train_dataset)}, Validation samples: {len(val_dataset)}")

    # --- Create DataLoaders ---
    train_loader = DataLoader(train_dataset, batch_size=config['batch_size'], collate_fn=collate_fn, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_dataset, batch_size=config['batch_size']*2, collate_fn=collate_fn, num_workers=0)

    # --- Sanity Check ---
    model_args = {'input_dim': 3484, 'num_classes': config['top_n'], 'hidden_dim': 384}
    if not sanity_check_overfit(StackedBiLSTMTransformerModel, model_args, train_loader, device):
        print("\n❌ Sanity check failed. Halting.")
        return

    # --- Full Training ---
    print(f"\n{'='*70}\n🚀 STARTING FULL TRAINING (30 EPOCHS)\n{'='*70}")
    model = StackedBiLSTMTransformerModel(**model_args).to(device)
    best_acc, best_f1, history = train_model(
        model, train_loader, val_loader, device, 
        epochs=config['epochs'], save_path='final_model.pth',
        num_classes=config['top_n']
    )
    
    print(f"\n🎉 ALL DONE!")
    print(f"   Best Validation Accuracy: {best_acc*100:.2f}%")
    print(f"   Best Validation F1-Score: {best_f1*100:.2f}%")

In [4]:
main()

Using device: cuda

📦 Loading dataset from D:\Balanced_20_Frames_Augmented\train_final.json
  ✅ Loaded global normalization stats.
  ✅ Found 5000 potential samples.

🔪 Splitting data into 80%/20% train/val sets...
   Train samples: 4000, Validation samples: 1000

🧪 SANITY CHECK: Attempting to overfit a single batch

🏗  Building StackedBiLSTMTransformerModel:
  Input dim: 3484, Hidden dim: 384, Output classes: 100
  LSTM Layers: 2, Transformer Layers: 1, Heads: 8
  Total parameters: 5,112,164
   Batch size: 6, Unique labels: 5
   Epoch  20: Loss=0.0359, Acc=100.00%

   ✅ SUCCESS! Overfitted in 20 epochs. Model can learn.

🚀 STARTING FULL TRAINING (30 EPOCHS)

🏗  Building StackedBiLSTMTransformerModel:
  Input dim: 3484, Hidden dim: 384, Output classes: 100
  LSTM Layers: 2, Transformer Layers: 1, Heads: 8
  Total parameters: 5,112,164

🚀 Epoch 1/30


Training: 100%|███████████████████████████████████████████████████| 125/125 [04:17<00:00,  2.06s/it]



📊 Train METRICS - Epoch 1

📈 Overall Metrics:
   Accuracy:           5.52%

   Macro Averages:
   - Precision:        2.39%
   - Recall:           3.03%

   Weighted Averages:
   - Precision:        4.06%
   - Recall:           5.52%

   - F1-Score (Macro): 2.55%
   - F1-Score (Weighted): 4.49%

📊 Confusion Matrix Statistics:
   True Positives:     20
   Total Predictions:  362


Val Evaluation: 100%|███████████████████████████████████████████████| 16/16 [01:06<00:00,  4.16s/it]



📊 Val METRICS - Epoch 1

📈 Overall Metrics:
   Accuracy:           10.53%

   Macro Averages:
   - Precision:        1.96%
   - Recall:           8.07%

   Weighted Averages:
   - Precision:        2.82%
   - Recall:           10.53%

   - F1-Score (Macro): 2.95%
   - F1-Score (Weighted): 4.14%

📊 Confusion Matrix Statistics:
   True Positives:     10
   Total Predictions:  95

📊 Epoch 1 Summary:
   Train -> Loss: 4.3472, Acc: 5.52%, F1: 2.55%
   Val   -> Loss: 3.4110, Acc: 10.53%, F1: 2.95%
✅ New best model saved! Val Acc: 10.53%, Val F1: 2.95%
   💾 Confusion matrix saved to: confusion_matrix_epoch_1.png

🚀 Epoch 2/30


...


📊 Val METRICS - Epoch 29

📈 Overall Metrics:
   Accuracy:           66.32%

   Macro Averages:
   - Precision:        68.95%
   - Recall:           71.96%

   Weighted Averages:
   - Precision:        70.92%
   - Recall:           66.32%

   - F1-Score (Macro): 67.30%
   - F1-Score (Weighted): 65.66%

📊 Confusion Matrix Statistics:
   True Positives:     63
   Total Predictions:  95

📊 Epoch 29 Summary:
   Train -> Loss: 0.5116, Acc: 87.02%, F1: 84.87%
   Val   -> Loss: 1.1482, Acc: 66.32%, F1: 67.30%

🚀 Epoch 30/30


Training: 100%|███████████████████████████████████████████████████| 125/125 [01:42<00:00,  1.22it/s]



📊 Train METRICS - Epoch 30

📈 Overall Metrics:
   Accuracy:           88.67%

   Macro Averages:
   - Precision:        87.41%
   - Recall:           87.04%

   Weighted Averages:
   - Precision:        87.97%
   - Recall:           88.67%

   - F1-Score (Macro): 86.79%
   - F1-Score (Weighted): 87.99%

📊 Confusion Matrix Statistics:
   True Positives:     321
   Total Predictions:  362


Val Evaluation: 100%|███████████████████████████████████████████████| 16/16 [00:24<00:00,  1.51s/it]



📊 Val METRICS - Epoch 30

📈 Overall Metrics:
   Accuracy:           76.84%

   Macro Averages:
   - Precision:        79.37%
   - Recall:           81.88%

   Weighted Averages:
   - Precision:        78.13%
   - Recall:           76.84%

   - F1-Score (Macro): 78.09%
   - F1-Score (Weighted): 75.58%

📊 Confusion Matrix Statistics:
   True Positives:     73
   Total Predictions:  95

📊 Epoch 30 Summary:
   Train -> Loss: 0.3041, Acc: 88.67%, F1: 86.79%
   Val   -> Loss: 0.6681, Acc: 76.84%, F1: 78.09%
   💾 Training history saved to: training_history.png

🏆 TRAINING COMPLETE!
   Best Validation Accuracy: 90.53%
   Best Validation F1-Score: 88.40%

🎉 ALL DONE!
   Best Validation Accuracy: 90.53%
   Best Validation F1-Score: 88.40%
